# IB Format Deepdive

| Field | Value |
|-------|-------|
| **Author** | Haewon |
| **Last update** | 2026-04-21 |
| **Objective** | Understand why KOR has disproportionately high IB format share (iOS: 42%, Android: 27%) — supply-side composition vs bidding strategy |
| **Scope** | KOR market (`req.country = 'KOR'`), L30D, iOS and Android separately |
| **Out of scope** | Advertiser-side genre split (deferred); bid price distribution via `pricing` table (optional) |
| **Key tables** | `fact_dsp_creative`, `bidrequest2026*`, `fact_supply` |
| **Additional context** | Follow-up to GDS × Sales workshop VT analysis (Jan 2026). Plan doc: `plans/ib_format_deepdive_plan.md` |

In [1]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [2]:
client = bigquery.Client(project="moloco-ods")

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

CHART_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'charts'))  # VT/charts/

## Section 1 — IB Share Benchmark (KOR vs Global)

Establish the baseline. Confirm and decompose the iOS 42% / Android 27% IB install share figures.
Compare KOR vs global average by OS using `fact_dsp_creative`.

In [5]:
# Top 5 countries by Moloco spend (L30D): USA, JPN, KOR, GBR, DEU — others bucketed
query_1 = """
WITH base AS (
    SELECT
        CASE
            WHEN campaign.country IN ('USA', 'JPN', 'KOR', 'GBR', 'DEU') THEN campaign.country
            ELSE 'Others'
        END AS geo,
        UPPER(campaign.os) AS os,
        creative.format AS cr_format,
        SUM(installs) AS installs
    FROM `moloco-ae-view.athena.fact_dsp_creative`
    WHERE date_utc >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
      AND UPPER(campaign.os) IN ('IOS', 'ANDROID')
    GROUP BY 1, 2, 3
)
SELECT
    geo,
    os,
    cr_format,
    installs,
    SUM(installs) OVER (PARTITION BY geo, os) AS total_installs,
    SAFE_DIVIDE(installs, SUM(installs) OVER (PARTITION BY geo, os)) AS install_share
FROM base
ORDER BY geo, os, installs DESC
"""
df_1 = run_query(query_1, label='Section 1 — IB share benchmark')
df_1.head(30)

✅ Section 1 — IB share benchmark: 121 rows


,geo,os,cr_format,installs,total_installs,install_share
0,DEU,ANDROID,vi,401677,791898,0.507233
1,DEU,ANDROID,ri,272519,791898,0.344134
2,DEU,ANDROID,ib,54924,791898,0.069357
3,DEU,ANDROID,nl,21724,791898,0.027433
4,DEU,ANDROID,ni,21176,791898,0.026741
5,DEU,ANDROID,nv,12386,791898,0.015641
6,DEU,ANDROID,ii,5929,791898,0.007487
7,DEU,ANDROID,vb,1410,791898,0.001781
8,DEU,ANDROID,db,142,791898,0.000179
9,DEU,ANDROID,di,11,791898,0.000014


In [7]:
# Filter to IB only; geo order: major markets first, Others last
GEO_ORDER = ['USA', 'JPN', 'KOR', 'GBR', 'DEU', 'Others']

df_1_ib = (
    df_1[df_1['cr_format'] == 'ib']
    .assign(
        install_share_pct=lambda d: d['install_share'] * 100,
        geo=lambda d: pd.Categorical(d['geo'], categories=GEO_ORDER, ordered=True)
    )
    .sort_values(['geo', 'os'])
)

fig = px.bar(
    df_1_ib,
    x='geo',
    y='install_share_pct',
    color='os',
    barmode='group',
    text=df_1_ib['install_share_pct'].map('{:.1f}%'.format),
    labels={'geo': 'Country', 'install_share_pct': 'IB Install Share (%)', 'os': 'OS'},
    title='IB Install Share % by Country × OS (L30D)',
    color_discrete_map={'IOS': '#636EFA', 'ANDROID': '#EF553B'},
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_ticksuffix='%', yaxis_range=[0, df_1_ib['install_share_pct'].max() * 1.2])
fig.show()
fig.write_image(os.path.join(CHART_DIR, 'ib_install_share_by_country_os.png'), scale=2)

## Section 2 — Supply Side: Bid Request Volume by Inventory Format

Test supply hypothesis: is Banner (`'B'`) inventory disproportionately more available in KOR vs globally?

- Table: `focal-elf-631.prod.bidrequest2026*`
- `inventory_format` values: `'B'` (Banner), `'N'` (Native), `'I'` (Interstitial)
- 1/10,000 sampled → multiply `COUNT(*)` by 10,000 for volume estimates
- Use `UPPER(os)` for OS filter

In [8]:
# Top 5 countries by Moloco spend (L30D): USA, JPN, KOR, GBR, DEU — others bucketed
query_2 = """
WITH base AS (
    SELECT
        CASE
            WHEN country IN ('USA', 'JPN', 'KOR', 'GBR', 'DEU') THEN country
            ELSE 'Others'
        END AS geo,
        UPPER(os) AS os,
        inventory_format,
        COUNT(*) * 10000 AS bid_requests_est
    FROM `focal-elf-631.prod.bidrequest2026*`
    WHERE DATE(timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
      AND inventory_format IN ('B', 'N', 'I')
      AND UPPER(os) IN ('IOS', 'ANDROID')
    GROUP BY 1, 2, 3
)
SELECT
    geo,
    os,
    inventory_format,
    bid_requests_est,
    SAFE_DIVIDE(bid_requests_est, SUM(bid_requests_est) OVER (PARTITION BY geo, os)) AS format_share
FROM base
ORDER BY geo, os, inventory_format
"""
df_2 = run_query(query_2, label='Section 2 — Supply format mix')
df_2.head(30)

✅ Section 2 — Supply format mix: 36 rows


,geo,os,inventory_format,bid_requests_est,format_share
0,DEU,ANDROID,B,183692820000,0.319158
1,DEU,ANDROID,I,297741270000,0.517312
2,DEU,ANDROID,N,94120570000,0.163530
3,DEU,IOS,B,193505370000,0.424910
4,DEU,IOS,I,172839720000,0.379531
5,DEU,IOS,N,89057750000,0.195558
6,GBR,ANDROID,B,122564890000,0.335887
7,GBR,ANDROID,I,176280860000,0.483095
8,GBR,ANDROID,N,66053540000,0.181019
9,GBR,IOS,B,263282280000,0.424795


In [10]:
# Stacked bar — bid request format mix (B/N/I) by country x OS
# One subplot per OS; x = geo, color = inventory_format (stacked to 100%)
GEO_ORDER = ['USA', 'JPN', 'KOR', 'GBR', 'DEU', 'Others']
FORMAT_LABELS = {'B': 'Banner', 'N': 'Native', 'I': 'Interstitial'}
FORMAT_COLORS = {'B': '#636EFA', 'N': '#EF553B', 'I': '#00CC96'}

df_2_plot = (
    df_2
    .assign(
        geo=lambda d: pd.Categorical(d['geo'], categories=GEO_ORDER, ordered=True),
        format_label=lambda d: d['inventory_format'].map(FORMAT_LABELS),
        format_share_pct=lambda d: d['format_share'] * 100
    )
    .sort_values(['geo', 'inventory_format'])
)

for os_val in ['IOS', 'ANDROID']:
    df_os = df_2_plot[df_2_plot['os'] == os_val]
    fig = px.bar(
        df_os,
        x='geo',
        y='format_share_pct',
        color='format_label',
        barmode='stack',
        text=df_os['format_share_pct'].map('{:.1f}%'.format),
        labels={'geo': 'Country', 'format_share_pct': 'Share of Bid Requests (%)', 'format_label': 'Format'},
        title=f'Bid Request Format Mix — {os_val} (L30D)',
        color_discrete_map={v: FORMAT_COLORS[k] for k, v in FORMAT_LABELS.items()},
        category_orders={'format_label': list(FORMAT_LABELS.values())},
    )
    fig.update_traces(textposition='inside', textfont_size=11)
    fig.update_layout(yaxis_ticksuffix='%', yaxis_range=[0, 100])
    fig.show()
    fig.write_image(os.path.join(CHART_DIR, f'bid_request_format_mix_{os_val}.png'), scale=2)

## Section 3 — Bidding Strategy: Bid Rate & Imp-to-Bid Ratio by Format

Test bidding hypothesis: does Moloco bid more aggressively on Banner in KOR vs globally?

- Table: `fact_supply`
- Canonical rates: `bid_rate = bids / bid_requests`, `win_rate = bids_won / bids`, `imp_to_bid = impressions / bids`
- **Gotcha:** do NOT filter `campaign_id IS NOT NULL` — zero-bid rows are required for accurate bid_rate
- Reference baseline (KakaoTalk × KAKAO, 180d): bid_rate 23.3%, win_rate 37.3%, recent trend: bid_rate ↓ win_rate ↑

In [11]:
# Using inventory_format (coarser) to preserve zero-bid rows for accurate bid_rate.
# cr_format is NULL for rows where Moloco did not bid — filtering it would inflate bid_rate.
# Top 5 countries by Moloco spend (L30D): USA, JPN, KOR, GBR, DEU — others bucketed
query_3 = """
SELECT
    CASE
        WHEN req.country IN ('USA', 'JPN', 'KOR', 'GBR', 'DEU') THEN req.country
        ELSE 'Others'
    END AS geo,
    UPPER(req.os) AS os,
    inventory_format,
    SUM(bid_requests) AS bid_requests,
    SUM(bids)         AS bids,
    SUM(bids_won)     AS bids_won,
    SUM(impressions)  AS impressions,
    SAFE_DIVIDE(SUM(bids),        NULLIF(SUM(bid_requests), 0)) AS bid_rate,
    SAFE_DIVIDE(SUM(bids_won),    NULLIF(SUM(bids), 0))         AS win_rate,
    SAFE_DIVIDE(SUM(impressions), NULLIF(SUM(bids_won), 0))     AS serve_rate,
    SAFE_DIVIDE(SUM(impressions), NULLIF(SUM(bids), 0))         AS imp_to_bid
FROM `moloco-ae-view.athena.fact_supply`
WHERE date_utc >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  AND UPPER(req.os) IN ('IOS', 'ANDROID')
  -- Do NOT add: AND campaign_id IS NOT NULL (excludes zero-bid rows → inflates bid_rate)
GROUP BY 1, 2, 3
ORDER BY 1, 2, bid_requests DESC
"""
df_3 = run_query(query_3, label='Section 3 — Bidding strategy by format')
df_3.head(30)

✅ Section 3 — Bidding strategy by format: 48 rows


,geo,os,inventory_format,bid_requests,bids,bids_won,impressions,bid_rate,win_rate,serve_rate,imp_to_bid
0,DEU,ANDROID,Video Interstitial,256392750000,9646112100,458000900,126142636,0.037622,0.047480,0.275420,0.013077
1,DEU,ANDROID,Banner,173556120000,7268776700,842785000,878669511,0.041881,0.115946,1.042578,0.120883
2,DEU,ANDROID,Native,89620970000,5347317400,1302378600,560401054,0.059666,0.243557,0.430290,0.104800
3,DEU,ANDROID,Interstitial,51642380000,2922079400,174217400,50679239,0.056583,0.059621,0.290897,0.017344
4,DEU,IOS,Banner,185228000000,20507655000,3941446100,3571554165,0.110716,0.192194,0.906153,0.174157
5,DEU,IOS,Video Interstitial,160774460000,7752369100,486555200,126261429,0.048219,0.062762,0.259501,0.016287
6,DEU,IOS,Native,82428570000,8739709000,1912340600,1512410989,0.106028,0.218811,0.790869,0.173050
7,DEU,IOS,Interstitial,24347920000,2915555600,246319200,45586208,0.119746,0.084484,0.185070,0.015636
8,GBR,ANDROID,Video Interstitial,159345440000,7413606500,385162900,111678870,0.046525,0.051954,0.289952,0.015064
9,GBR,ANDROID,Banner,115333570000,6291427300,776205300,657694389,0.054550,0.123375,0.847320,0.104538


In [12]:
# Grouped bar — bid_rate / win_rate / imp_to_bid by inventory_format x country, per OS
# Melt metrics into long form for faceted chart
METRICS = {'bid_rate': 'Bid Rate', 'win_rate': 'Win Rate', 'imp_to_bid': 'Imp-to-Bid'}
GEO_ORDER = ['USA', 'JPN', 'KOR', 'GBR', 'DEU', 'Others']

df_3_plot = (
    df_3
    .assign(geo=lambda d: pd.Categorical(d['geo'], categories=GEO_ORDER, ordered=True))
    .melt(
        id_vars=['geo', 'os', 'inventory_format'],
        value_vars=list(METRICS.keys()),
        var_name='metric',
        value_name='rate'
    )
    .assign(
        metric_label=lambda d: d['metric'].map(METRICS),
        rate_pct=lambda d: d['rate'] * 100
    )
)

for os_val in ['IOS', 'ANDROID']:
    df_os = df_3_plot[df_3_plot['os'] == os_val]
    fig = px.bar(
        df_os,
        x='geo',
        y='rate_pct',
        color='inventory_format',
        facet_col='metric_label',
        barmode='group',
        labels={'geo': 'Country', 'rate_pct': 'Rate (%)', 'inventory_format': 'Format'},
        title=f'Bidding Strategy by Format — {os_val} (L30D)',
        category_orders={'metric_label': list(METRICS.values())},
    )
    fig.update_layout(yaxis_ticksuffix='%')
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show()
    fig.write_image(os.path.join(CHART_DIR, f'bidding_strategy_by_format_{os_val}.png'), scale=2)

## Section 4-A — IB Deep Dive: Gaming vs Non-Gaming Publisher Inventory (KOR)

Are IB ads in KOR concentrated in gaming publisher inventory?

- Table: `fact_supply`, filter: `cr_format = 'ib'`, `req.country = 'KOR'`
- Split: `req.app_is_gaming` (BOOL)
- **Note:** `req.app_is_gaming` classifies the **publisher app** (supply side), not the advertiser's game
- Metrics: bid_rate, win_rate, imp_to_bid, avg bid price (`bid_price_usd / bids`)

In [13]:
query_4a = """
SELECT
    req.app_is_gaming,
    UPPER(req.os) AS os,
    SUM(bid_requests) AS bid_requests,
    SUM(bids)         AS bids,
    SUM(bids_won)     AS bids_won,
    SUM(impressions)  AS impressions,
    SAFE_DIVIDE(SUM(bids),        NULLIF(SUM(bid_requests), 0)) AS bid_rate,
    SAFE_DIVIDE(SUM(bids_won),    NULLIF(SUM(bids), 0))         AS win_rate,
    SAFE_DIVIDE(SUM(impressions), NULLIF(SUM(bids), 0))         AS imp_to_bid,
    SAFE_DIVIDE(SUM(bid_price_usd), NULLIF(SUM(bids), 0))       AS avg_bid_price_usd
FROM `moloco-ae-view.athena.fact_supply`
WHERE date_utc >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  AND req.country = 'KOR'
  AND cr_format = 'ib'
  AND UPPER(req.os) IN ('IOS', 'ANDROID')
  -- Do NOT add: AND campaign_id IS NOT NULL
GROUP BY 1, 2
ORDER BY 2, 1
"""
df_4a = run_query(query_4a, label='Section 4-A — IB gaming vs non-gaming publisher')
df_4a

✅ Section 4-A — IB gaming vs non-gaming publisher: 6 rows


,app_is_gaming,os,bid_requests,bids,bids_won,impressions,bid_rate,win_rate,imp_to_bid,avg_bid_price_usd
0,<NA>,ANDROID,6814810000,387754800,57586600,158140718,0.056899,0.148513,0.407837,0.000198
1,False,ANDROID,153080630000,13674165000,4530462600,4139155157,0.089327,0.331315,0.302699,0.000248
2,True,ANDROID,66042370000,4390600800,719282900,631685706,0.066482,0.163823,0.143872,0.000144
3,<NA>,IOS,978710000,105334500,40070300,106441366,0.107626,0.380410,1.010508,0.000109
4,False,IOS,50993870000,11574639000,5568939600,4026945549,0.226981,0.481133,0.347911,0.000193
5,True,IOS,40924660000,5774026400,1624664800,1534965202,0.141089,0.281375,0.265840,0.000097


In [14]:
# Grouped bar — bid_rate / win_rate / imp_to_bid by publisher gaming flag x OS
# Note: req.app_is_gaming = publisher app genre (supply side), not advertiser vertical
METRICS = {'bid_rate': 'Bid Rate', 'win_rate': 'Win Rate', 'imp_to_bid': 'Imp-to-Bid'}

df_4a_plot = (
    df_4a
    .assign(
        gaming_label=lambda d: d['app_is_gaming'].map({True: 'Gaming', False: 'Non-Gaming'})
    )
    .melt(
        id_vars=['gaming_label', 'os'],
        value_vars=list(METRICS.keys()),
        var_name='metric',
        value_name='rate'
    )
    .assign(
        metric_label=lambda d: d['metric'].map(METRICS),
        rate_pct=lambda d: d['rate'] * 100
    )
)

for os_val in ['IOS', 'ANDROID']:
    df_os = df_4a_plot[df_4a_plot['os'] == os_val]
    fig = px.bar(
        df_os,
        x='gaming_label',
        y='rate_pct',
        color='gaming_label',
        facet_col='metric_label',
        barmode='group',
        text=df_os['rate_pct'].map('{:.1f}%'.format),
        labels={'gaming_label': '', 'rate_pct': 'Rate (%)', 'metric_label': ''},
        title=f'IB Funnel — Gaming vs Non-Gaming Publisher ({os_val}, KOR, L30D)',
        color_discrete_map={'Gaming': '#636EFA', 'Non-Gaming': '#EF553B'},
        category_orders={'metric_label': list(METRICS.values())},
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis_ticksuffix='%', showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show()
    fig.write_image(os.path.join(CHART_DIR, f'ib_gaming_vs_nongaming_{os_val}.png'), scale=2)

# Avg bid price table
print('\nAvg bid price (USD):')
display(
    df_4a.assign(gaming_label=lambda d: d['app_is_gaming'].map({True: 'Gaming', False: 'Non-Gaming'}))
    [['gaming_label', 'os', 'bid_requests', 'bids', 'bids_won', 'impressions',
      'bid_rate', 'win_rate', 'imp_to_bid', 'avg_bid_price_usd']]
    .sort_values(['os', 'gaming_label'])
    .style.format({
        'bid_rate': '{:.1%}', 'win_rate': '{:.1%}', 'imp_to_bid': '{:.3f}',
        'avg_bid_price_usd': '${:.4f}',
        'bid_requests': '{:,.0f}', 'bids': '{:,.0f}', 'bids_won': '{:,.0f}', 'impressions': '{:,.0f}'
    })
)


Avg bid price (USD):


,gaming_label,os,bid_requests,bids,bids_won,impressions,bid_rate,win_rate,imp_to_bid,avg_bid_price_usd
2,Gaming,ANDROID,"66,042,370,000","4,390,600,800","719,282,900","631,685,706",6.6%,16.4%,0.144,$0.0001
1,Non-Gaming,ANDROID,"153,080,630,000","13,674,165,000","4,530,462,600","4,139,155,157",8.9%,33.1%,0.303,$0.0002
0,nan,ANDROID,"6,814,810,000","387,754,800","57,586,600","158,140,718",5.7%,14.9%,0.408,$0.0002
5,Gaming,IOS,"40,924,660,000","5,774,026,400","1,624,664,800","1,534,965,202",14.1%,28.1%,0.266,$0.0001
4,Non-Gaming,IOS,"50,993,870,000","11,574,639,000","5,568,939,600","4,026,945,549",22.7%,48.1%,0.348,$0.0002
3,nan,IOS,"978,710,000","105,334,500","40,070,300","106,441,366",10.8%,38.0%,1.011,$0.0001


## Section 4-B — IB Deep Dive: Kakao vs Non-Kakao (KOR)

Is IB bidding behavior in KOR driven by KakaoTalk inventory?

- Table: `fact_supply`, filter: `cr_format = 'ib'`, `req.country = 'KOR'`
- Split: `exchange = 'KAKAO'` vs others
- Metrics: bid_rate, win_rate, imp_to_bid, avg bid price (`bid_price_usd / bids`)

In [15]:
query_4b = """
SELECT
    CASE WHEN exchange = 'KAKAO' THEN 'Kakao' ELSE 'Non-Kakao' END AS exchange_group,
    UPPER(req.os) AS os,
    SUM(bid_requests) AS bid_requests,
    SUM(bids)         AS bids,
    SUM(bids_won)     AS bids_won,
    SUM(impressions)  AS impressions,
    SAFE_DIVIDE(SUM(bids),        NULLIF(SUM(bid_requests), 0)) AS bid_rate,
    SAFE_DIVIDE(SUM(bids_won),    NULLIF(SUM(bids), 0))         AS win_rate,
    SAFE_DIVIDE(SUM(impressions), NULLIF(SUM(bids), 0))         AS imp_to_bid,
    SAFE_DIVIDE(SUM(bid_price_usd), NULLIF(SUM(bids), 0))       AS avg_bid_price_usd
FROM `moloco-ae-view.athena.fact_supply`
WHERE date_utc >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  AND req.country = 'KOR'
  AND cr_format = 'ib'
  AND UPPER(req.os) IN ('IOS', 'ANDROID')
  -- Do NOT add: AND campaign_id IS NOT NULL
GROUP BY 1, 2
ORDER BY 2, 1
"""
df_4b = run_query(query_4b, label='Section 4-B — IB Kakao vs Non-Kakao')
df_4b

✅ Section 4-B — IB Kakao vs Non-Kakao: 4 rows


,exchange_group,os,bid_requests,bids,bids_won,impressions,bid_rate,win_rate,imp_to_bid,avg_bid_price_usd
0,Kakao,ANDROID,39937960000,8077911100,3291974400,2615706193,0.202261,0.407528,0.323810,0.000216
1,Non-Kakao,ANDROID,185999850000,10374609500,2015357700,2313275388,0.055778,0.194259,0.222975,0.000227
2,Kakao,IOS,20930080000,6624467400,4091881500,2844802712,0.316505,0.617692,0.429439,0.000191
3,Non-Kakao,IOS,71967160000,10829532500,3141793200,2823549405,0.150479,0.290113,0.260727,0.000143


In [16]:
# Grouped bar — bid_rate / win_rate / imp_to_bid by Kakao vs Non-Kakao x OS
METRICS = {'bid_rate': 'Bid Rate', 'win_rate': 'Win Rate', 'imp_to_bid': 'Imp-to-Bid'}

df_4b_plot = (
    df_4b
    .melt(
        id_vars=['exchange_group', 'os'],
        value_vars=list(METRICS.keys()),
        var_name='metric',
        value_name='rate'
    )
    .assign(
        metric_label=lambda d: d['metric'].map(METRICS),
        rate_pct=lambda d: d['rate'] * 100
    )
)

for os_val in ['IOS', 'ANDROID']:
    df_os = df_4b_plot[df_4b_plot['os'] == os_val]
    fig = px.bar(
        df_os,
        x='exchange_group',
        y='rate_pct',
        color='exchange_group',
        facet_col='metric_label',
        barmode='group',
        text=df_os['rate_pct'].map('{:.1f}%'.format),
        labels={'exchange_group': '', 'rate_pct': 'Rate (%)', 'metric_label': ''},
        title=f'IB Funnel — Kakao vs Non-Kakao ({os_val}, KOR, L30D)',
        color_discrete_map={'Kakao': '#FFA15A', 'Non-Kakao': '#AB63FA'},
        category_orders={'metric_label': list(METRICS.values())},
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis_ticksuffix='%', showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show()
    fig.write_image(os.path.join(CHART_DIR, f'ib_kakao_vs_nonkakao_{os_val}.png'), scale=2)

# Avg bid price + summary table
print('\nIB funnel summary (KOR):')
display(
    df_4b[['exchange_group', 'os', 'bid_requests', 'bids', 'bids_won', 'impressions',
           'bid_rate', 'win_rate', 'imp_to_bid', 'avg_bid_price_usd']]
    .sort_values(['os', 'exchange_group'])
    .style.format({
        'bid_rate': '{:.1%}', 'win_rate': '{:.1%}', 'imp_to_bid': '{:.3f}',
        'avg_bid_price_usd': '${:.4f}',
        'bid_requests': '{:,.0f}', 'bids': '{:,.0f}', 'bids_won': '{:,.0f}', 'impressions': '{:,.0f}'
    })
)


IB funnel summary (KOR):


,exchange_group,os,bid_requests,bids,bids_won,impressions,bid_rate,win_rate,imp_to_bid,avg_bid_price_usd
0,Kakao,ANDROID,"39,937,960,000","8,077,911,100","3,291,974,400","2,615,706,193",20.2%,40.8%,0.324,$0.0002
1,Non-Kakao,ANDROID,"185,999,850,000","10,374,609,500","2,015,357,700","2,313,275,388",5.6%,19.4%,0.223,$0.0002
2,Kakao,IOS,"20,930,080,000","6,624,467,400","4,091,881,500","2,844,802,712",31.7%,61.8%,0.429,$0.0002
3,Non-Kakao,IOS,"71,967,160,000","10,829,532,500","3,141,793,200","2,823,549,405",15.0%,29.0%,0.261,$0.0001
